# Minimal working example using BERT (`distBERT` model)

In [1]:
!pip install pandas torch transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 29.3 MB/s eta 0:00:00


## Setup

In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
import faiss

## Pseudo data creation

In [3]:
# ----------------------------
# 1) Sample pseudo "data"
# ----------------------------
items = pd.DataFrame([
    {"item_id":"i1","title":"Noise Cancelling Headphones","description":"Wireless noise-cancelling headphones with 30-hour battery life","category":"electronics"},
    {"item_id":"i2","title":"Mechanical Keyboard","description":"Mechanical keyboard with RGB and hot-swappable switches","category":"electronics"},
    {"item_id":"i3","title":"Running Shoes","description":"Running shoes designed for long distance comfort and stability","category":"sports"},
    {"item_id":"i4","title":"Vegetarian Cookbook","description":"Cookbook featuring quick vegetarian recipes for busy weeknights","category":"books"},
    {"item_id":"i5","title":"Fitness Smartwatch","description":"Smartwatch with heart rate monitoring, GPS, and sleep tracking","category":"electronics"},
])

events = pd.DataFrame([
    {"user_id":"u1","item_id":"i1","ts":"2026-01-10"},
    {"user_id":"u1","item_id":"i5","ts":"2026-01-11"},
    {"user_id":"u2","item_id":"i2","ts":"2026-01-12"},
    {"user_id":"u3","item_id":"i3","ts":"2026-01-13"},
    {"user_id":"u3","item_id":"i4","ts":"2026-01-14"},
])
events["ts"] = pd.to_datetime(events["ts"])


In [49]:
items_large = pd.read_csv("items.csv")
events_large = pd.read_csv("events.csv")
events_large.head()


,user_id,item_id,ts,event_type,session_id,device,dwell_seconds,price_at_event,category_at_event
0,u1,i2,2026-01-01 07:47:00,click,s_u1_20260101_1,tablet,56,129.00,electronics
1,u1,i13,2026-01-04 08:01:00,view,s_u1_20260104_2,tablet,7,149.99,electronics
2,u1,i2,2026-01-07 14:28:00,click,s_u1_20260107_3,mobile,72,129.00,electronics
3,u1,i13,2026-01-07 19:06:00,view,s_u1_20260107_1,tablet,67,149.99,electronics
4,u1,i1,2026-01-10 08:42:00,add_to_cart,s_u1_20260110_1,desktop,46,199.99,electronics


In [56]:
events_df = events_large[['user_id', 'item_id', 'ts', 'dwell_seconds']].copy()
items_df = items_large.copy()
events_df = events_df[events_df['dwell_seconds'] > 50]


## BERT encoder (What makes BERT work?)

In [57]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "distilbert-base-uncased"  # for illustration, 66M model

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
encoder = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


**Evaluate the BERT encoder**

In [58]:
# ----------------------------
# 2) BERT encode helper
# ----------------------------

# turn off gradient calculation (no back propagation in using BERT)
@torch.no_grad()
def bert_embed(texts, max_len=768):
    batch = tokenizer(
        texts, padding=True, truncation=True, max_length=max_len, return_tensors="pt"
    )
    batch = {k: v.to(DEVICE) for k, v in batch.items()}
    out = encoder(**batch)
    cls = out.last_hidden_state[:, 0]          # first column [CLS]-like token for classification
    emb = F.normalize(cls, dim=-1)             # normalization
    return emb.cpu().numpy().astype("float32") # size (B, 768)


In [24]:
items_df.head()

,item_id,title,description,category,brand,price,tags
0,i1,Noise Cancelling Headphones,Wireless noise-cancelling headphones with 30-h...,electronics,SoundPeak,199.99,"audio,wireless,travel,premium"
1,i2,Mechanical Keyboard,"Mechanical keyboard with RGB backlighting, hot...",electronics,KeyForge,129.00,"keyboard,gaming,productivity,desk"
2,i3,Running Shoes,Running shoes designed for long distance comfo...,sports,StrideLab,110.00,"running,fitness,outdoors,comfort"
3,i4,Vegetarian Cookbook,"Cookbook featuring quick vegetarian recipes, p...",books,HomeTable Press,24.99,"cooking,vegetarian,recipes,home"
4,i5,Fitness Smartwatch,"Smartwatch with heart rate monitoring, GPS, sl...",electronics,PulsePath,249.00,"wearables,fitness,gps,health"


## Embedding items into vectors (for comparisons)

In [59]:
# ----------------------------
# 3) Offline job: item embeddings
# ----------------------------

items["text"] = items["title"] + ". " + items["description"]

item_vecs = bert_embed(items["text"].tolist()) # the actual embedding process
print(items['text'])
item_id_list = items["item_id"].tolist()

# Build ANN index (inner product works with normalized vectors)
index = faiss.IndexFlatIP(item_vecs.shape[1])
index.add(item_vecs)


0    Noise Cancelling Headphones. Wireless noise-ca...
1    Mechanical Keyboard. Mechanical keyboard with ...
2    Running Shoes. Running shoes designed for long...
3    Vegetarian Cookbook. Cookbook featuring quick ...
4    Fitness Smartwatch. Smartwatch with heart rate...
Name: text, dtype: object


In [60]:
# Convert all components to string, specifically handling 'price' with .astype(str)
items_df["text"] = items_df["title"] + ". " + \
                    items_df["description"] + ". " + \
                    items_df["category"] + ". " + \
                    items_df["brand"] + ". " + \
                    items_df["tags"] + ". " + \
                    items_df["price"].astype(str)

# Re-embed using the larger items_df
item_vecs = bert_embed(items_df["text"].tolist())
item_id_list = items_df["item_id"].tolist()

# Re-build ANN index with the new embeddings
index = faiss.IndexFlatIP(item_vecs.shape[1])
index.add(item_vecs)


## What user-specific data are there?

In [61]:
# ----------------------------
# 4) Feature builder: user text from last N clicks
# ----------------------------
def build_user_text(user_id, events, items, N=3):
    hist = (events[events["user_id"] == user_id]
            .sort_values("ts")
            .tail(N)["item_id"]
            .tolist())
    if not hist:
        return "no history"
    text = items.set_index("item_id").loc[hist, "text"].tolist()
    return " ".join(text), set(hist)


## How to recommend "similar" item with BERT?

In [62]:
# ----------------------------
# 5) "What to recommend" function
# ----------------------------
def recommend(user_id, k=3):
    # Changed 'n=3' to 'N=3' to match the function signature
    user_text, seen = build_user_text(user_id, events_df, items_df, N=3)
    u = bert_embed([user_text])  # (1, 768)
    scores, idx = index.search(u, k + len(seen))  # keep track of what was seen by the user
    recs = []
    for j in idx[0]:
        iid = item_id_list[j]
        if iid not in seen:
            recs.append(iid)
        if len(recs) == k:
            break
    return recs

In [63]:
for u in ["u1","u2","u3"]:
    print(u, "->", recommend(u, k=3))


u1 -> ['i13', 'i6', 'i14']
u2 -> ['i18', 'i13', 'i17']
u3 -> ['i6', 'i18', 'i5']


In [64]:
events_df[events_df['user_id'] == 'u1']['item_id'].value_counts()

,count
item_id,
i2,4
i13,3
i3,3
i6,2
i19,2
i17,1
i15,1
i11,1
i14,1


In [74]:
items_df[items_df['item_id'] == 'i19']

,item_id,title,description,category,brand,price,tags,text
18,i19,Cycling Gloves,Breathable cycling gloves with gel padding for...,sports,TrailSip,27.5,"cycling,fitness,outdoors,comfort",Cycling Gloves. Breathable cycling gloves with...


In [67]:
items_df[items_df['item_id'] == 'i13']

,item_id,title,description,category,brand,price,tags,text
12,i13,Wireless Earbuds,"Pocket-sized wireless earbuds with ANC, transp...",electronics,SoundPeak,149.99,"audio,wireless,fitness,commute",Wireless Earbuds. Pocket-sized wireless earbud...


In [68]:
items_df[items_df['item_id'] == 'i6']

,item_id,title,description,category,brand,price,tags,text
5,i6,Yoga Mat,Non-slip yoga mat with extra cushioning for ho...,sports,FlexRoot,39.99,"yoga,fitness,home,wellness",Yoga Mat. Non-slip yoga mat with extra cushion...


In [69]:
items_df[items_df['item_id'] == 'i14']

,item_id,title,description,category,brand,price,tags,text
13,i14,Resistance Bands Set,Five-level resistance bands set for strength t...,sports,FlexRoot,22.0,"fitness,strength,home,training",Resistance Bands Set. Five-level resistance ba...
